# Inference H2 guarded — một prompt mỗi mẫu
Đọc các run trong `exports/v3/manifest.json`, không dựng lại evidence.
Chọn kernel **GRACE H2 Ubuntu**. Chạy các ô theo thứ tự. Ô gọi API chỉ thực hiện khi `EXECUTE = True`.
FPT dùng file `.env` chung ở thư mục LabThayTho: `FPT_API`, `FPT_MODEL`, tùy chọn `FPT_BASE_URL`. Vertex dùng ADC và `GOOGLE_CLOUD_PROJECT`.
Mỗi mẫu chỉ được gửi tối đa một lần. Mẫu lỗi hoặc bị ngắt sau khi đã đặt chỗ không tự gửi lại, để tránh gọi trùng. Chỉ đánh giá sau khi hoàn tất inference.

In [1]:
from pathlib import Path
import sys, json
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "grace_h2").is_dir()), None)
if ROOT is None:
    ROOT = Path("/media/quang-dung/Data/JUNIOR/LabThayTho/GRACE-H2-GuardedEvidence")
assert (ROOT / "grace_h2").is_dir()
sys.path.insert(0, str(ROOT))
from grace_h2.inference import infer, provider_spec
from grace_h2.evaluation import evaluate
from grace_h2.store import Store
manifest = json.loads((ROOT / "exports/v3/manifest.json").read_text())
RUNS = {ds: ROOT / "runs/v3" / ds / entry["run_id"] for ds, entry in manifest["datasets"].items()}
print("Project:", ROOT)
print("Runs:", {ds: str(path) for ds, path in RUNS.items()})

Project: /media/quang-dung/Data/JUNIOR/LabThayTho/GRACE-H2-GuardedEvidence
Runs: {'devign': '/media/quang-dung/Data/JUNIOR/LabThayTho/GRACE-H2-GuardedEvidence/runs/v3/devign/a6dae6c1d9533d815bd7', 'reveal': '/media/quang-dung/Data/JUNIOR/LabThayTho/GRACE-H2-GuardedEvidence/runs/v3/reveal/24867fce07a449ee99a0'}


## 1. Chọn dịch vụ và dữ liệu
Mặc định chạy 50% mẫu đầu mỗi dataset, làm tròn xuống. Đổi PERCENT thành 100 để chạy nốt; lượt đã ghi nhận tự động bỏ qua. Đổi thành `DATASETS = ["devign", "reveal"]` để chạy cả hai.
`PROVIDER = "fpt"` dùng model trong `.env`; `"vertex"` dùng model của contract.
FPT có thể khác model baseline nên không mặc nhiên là so sánh cùng điều kiện.

In [2]:
PROVIDER = "fpt"  # hoặc "vertex"
DATASETS = ["devign", "reveal"]
PERCENT = 50  # phần trăm mẫu đầu mỗi dataset; đổi 100 để chạy nốt
EXECUTE = True  # đổi True khi muốn bắt đầu gọi API
assert PROVIDER in ("fpt", "vertex")
assert DATASETS and len(DATASETS) == len(set(DATASETS)) and all(ds in RUNS for ds in DATASETS)
settings = provider_spec(ROOT, PROVIDER)  # không in API key
print(json.dumps(settings, ensure_ascii=False, indent=2))
for ds in DATASETS:
    print(ds, infer(ROOT, RUNS[ds], provider=PROVIDER, execute=False, percent=PERCENT))

{
  "provider": "fpt",
  "temperature": 0.0,
  "seed": 42,
  "max_output_tokens": 256,
  "max_attempts": 1,
  "model": "DeepSeek-V4-Flash",
  "base_url": "https://token-api.fpt.ai/v1",
  "comparison_note": "Provider/model differs from Vertex contract; do not claim a matched comparison automatically."
}
devign {'queries': 2732, 'percent': 50, 'selected_queries': 1366, 'reserved_or_completed': 0, 'remaining_max_calls': 1366, 'prompts_per_query': 1, 'max_attempts': 1, 'provider': 'fpt', 'execute': False}
reveal {'queries': 2274, 'percent': 50, 'selected_queries': 1137, 'reserved_or_completed': 0, 'remaining_max_calls': 1137, 'prompts_per_query': 1, 'max_attempts': 1, 'provider': 'fpt', 'execute': False}


## 2. Gọi LLM
Sau khi kiểm tra model và số lượt còn lại ở trên, đặt `EXECUTE = True`, chạy lại ô cấu hình rồi chạy ô này.
Kết quả được ghi ngay vào `run.sqlite3` của từng run. Hiển thị tiến độ mỗi 25 mẫu.
Nếu dừng kernel, mở lại và chạy lại các ô: mẫu đã được đặt chỗ/hoàn tất sẽ được bỏ qua.

In [3]:
if not EXECUTE:
    print("Chưa gọi API. Đặt EXECUTE = True trong ô cấu hình để chạy.")
else:
    for ds in DATASETS:
        print("Bắt đầu:", ds, flush=True)
        print(infer(ROOT, RUNS[ds], provider=PROVIDER, execute=True, percent=PERCENT))

Bắt đầu: devign
inference 25/1366, elapsed 63.2s
inference 50/1366, elapsed 119.1s
inference 75/1366, elapsed 144.1s
inference 100/1366, elapsed 234.2s
inference 125/1366, elapsed 372.4s
inference 150/1366, elapsed 445.4s
inference 175/1366, elapsed 500.8s
inference 200/1366, elapsed 605.4s
inference 225/1366, elapsed 717.5s
inference 250/1366, elapsed 830.9s
inference 275/1366, elapsed 968.8s
inference 300/1366, elapsed 1159.2s
inference 325/1366, elapsed 1257.8s
inference 350/1366, elapsed 1340.0s
inference 375/1366, elapsed 1445.4s
inference 400/1366, elapsed 1526.3s
inference 425/1366, elapsed 1624.2s
inference 450/1366, elapsed 1692.3s
inference 475/1366, elapsed 1768.3s
inference 500/1366, elapsed 1835.3s
inference 525/1366, elapsed 1856.4s
inference 550/1366, elapsed 1906.8s
inference 575/1366, elapsed 1927.1s
inference 600/1366, elapsed 1947.8s
inference 625/1366, elapsed 1982.2s
inference 650/1366, elapsed 2004.0s
inference 675/1366, elapsed 2043.3s
inference 700/1366, elapsed

KeyboardInterrupt: 

## 3. Kiểm tra trạng thái và đánh giá
Ô này báo số mẫu hoàn tất, lỗi và lượt gửi chưa xác định. Chỉ khi tất cả mẫu có kết quả lưu mới đọc nhãn test để đánh giá. Kết quả ghi vào `evaluation.json` trong thư mục run; lỗi/đầu ra không hợp lệ được báo riêng.

In [ ]:
for ds in DATASETS:
    with Store(RUNS[ds]) as store:
        total = store.meta("report")["queries"]
        states = dict(store.db.execute("SELECT state, COUNT(*) FROM responses GROUP BY state"))
    print(ds, "total:", total, "states:", states)
    if states.get("complete", 0) == total:
        result = evaluate(ROOT, RUNS[ds])
        print(json.dumps(result, ensure_ascii=False, indent=2))
    else:
        print("Chưa đủ kết quả hoàn tất; chưa đánh giá.")